# =============================================================================
# UPI TRANSACTIONS CDC FEED PROJECT - ENHANCED MOCK DATA GENERATOR
# =============================================================================
# Generates realistic UPI transaction data with CDC operations
# INSERT / UPDATE / DELETE patterns using Delta Lake
# =============================================================================


In [ ]:
import random
from datetime import datetime, timedelta
import time

from pyspark.sql import functions as F
from delta.tables import DeltaTable

catalog_name = "`upi_transactions`"
schema_name = "default"
raw_table = f"{catalog_name}.{schema_name}.raw_upi_transactions_v1"


In [ ]:
MERCHANTS = [
    {"merchant_id": "M001", "merchant_name": "Amazon India", "merchant_category": "E-commerce"},
    {"merchant_id": "M002", "merchant_name": "Swiggy", "merchant_category": "Food Delivery"},
    {"merchant_id": "M003", "merchant_name": "Uber", "merchant_category": "Transportation"},
    {"merchant_id": "M004", "merchant_name": "Netflix", "merchant_category": "Entertainment"},
    {"merchant_id": "M005", "merchant_name": "BigBasket", "merchant_category": "Grocery"},
    {"merchant_id": "M006", "merchant_name": "Flipkart", "merchant_category": "E-commerce"},
    {"merchant_id": "M007", "merchant_name": "Zomato", "merchant_category": "Food Delivery"},
    {"merchant_id": "M008", "merchant_name": "Ola", "merchant_category": "Transportation"}
]

UPI_IDS = ["user123@paytm", "user456@phonepe", "user789@googlepay"]
CUSTOMER_IDS = ["CUST001", "CUST002", "CUST003"]
PAYMENT_METHODS = ["UPI", "QR Code"]
DEVICE_TYPES = ["Mobile", "Tablet"]
OPERATING_SYSTEMS = ["Android", "iOS"]

transaction_counter = 1


In [ ]:
def insert_new_transactions(num_transactions=5):
    global transaction_counter
    rows = []

    for _ in range(num_transactions):
        merchant = random.choice(MERCHANTS)
        amount = round(random.uniform(50, 5000), 2)
        txn_id = f"TXN_{datetime.now().strftime('%Y%m%d')}_{transaction_counter:06d}"
        transaction_counter += 1

        rows.append((
            txn_id,
            random.choice(UPI_IDS),
            merchant["merchant_id"],
            merchant["merchant_name"],
            merchant["merchant_category"],
            float(amount),
            "INR",
            datetime.now(),
            "completed",
            random.choice(PAYMENT_METHODS),
            random.choice(DEVICE_TYPES),
            random.choice(OPERATING_SYSTEMS),
            "v1.0.0",
            18.5204,
            73.8567,
            "Pune",
            "Maharashtra",
            "India",
            random.choice(CUSTOMER_IDS),
            "26-35",
            "Male",
            float(amount * 0.005),
            float(amount * 0.01),
            datetime.now(),
            datetime.now()
        ))

    df = spark.createDataFrame(rows, spark.table(raw_table).schema)
    df.write.format("delta").mode("append").saveAsTable(raw_table)


In [ ]:
def update_existing_transactions(num_updates=3):
    df = spark.sql(f"SELECT * FROM {raw_table} ORDER BY created_at DESC LIMIT {num_updates}")
    if df.count() == 0:
        return

    updated_df = df.withColumn(
        "transaction_amount",
        F.col("transaction_amount") * F.lit(1.1)
    ).withColumn(
        "updated_at",
        F.current_timestamp()
    )

    DeltaTable.forName(spark, raw_table).alias("t") \
        .merge(updated_df.alias("s"), "t.transaction_id = s.transaction_id") \
        .whenMatchedUpdateAll() \
        .execute()


In [ ]:
def delete_transactions(num_deletes=2):
    ids = spark.sql(f"SELECT transaction_id FROM {raw_table} ORDER BY created_at DESC LIMIT {num_deletes}").collect()
    delta = DeltaTable.forName(spark, raw_table)
    for r in ids:
        delta.delete(f"transaction_id = '{r.transaction_id}'")


In [ ]:
def continuous_cdc_data_generation(duration_minutes=20):
    end_time = datetime.now() + timedelta(minutes=duration_minutes)
    while datetime.now() < end_time:
        op = random.choice(["INSERT", "UPDATE", "DELETE"])
        if op == "INSERT":
            insert_new_transactions(5)
        elif op == "UPDATE":
            update_existing_transactions(2)
        else:
            delete_transactions(1)
        time.sleep(120)
